In [ ]:
from langchain_core.runnables import RunnableConfig
from typing import List, Optional, Dict, Any, TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph
from langgraph.graph import START, END
from langgraph.runtime import Runtime

In [ ]:
# Define state
class BMI_Class(TypedDict):
    bmi: float
    height: float # in meters
    weight: float # in kilograms
    bmi_category: str # e.g., 'Underweight', 'Normal weight', 'Overweight', 'Obesity'

In [ ]:
# Create a node to calculate_bmi
def fun_calculate_bmi(state: BMI_Class) -> BMI_Class:

    height = state['height']
    weight = state['weight']

    bmi = weight / (height ** 2)
    state['bmi'] = round(bmi, 2)

    return state

In [ ]:
# Create a node to categorize_bmi
def fun_categorize_bmi(state: BMI_Class) -> BMI_Class:
    bmi = state['bmi']

    if bmi < 18.5:
        state['bmi_category'] = 'Underweight'
    elif 18.5 <= bmi < 24.9:
        state['bmi_category'] = 'Normal weight'
    elif 25 <= bmi < 29.9:
        state['bmi_category'] = 'Overweight'
    else:
        state['bmi_category'] = 'Obesity'

    return state

In [ ]:
# Define Graph
graph = StateGraph(BMI_Class)

In [ ]:
# add node to Graph
graph.add_node('calculate_bmi', fun_calculate_bmi)
graph.add_node('categorize_bmi', fun_categorize_bmi)


In [ ]:
# add edge to graph
graph.add_edge(START, 'calculate_bmi')
graph.add_edge('calculate_bmi', 'categorize_bmi')
graph.add_edge('categorize_bmi', END)

In [ ]:
# Compile Graph
workflow = graph.compile()

In [ ]:
# Execute Graph

initial_state = {
    'height': 1.75,
    'weight': 70
}

final_state = workflow.invoke(initial_state)

print(final_state)

In [ ]:
# Visualize Graph
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())